In [ ]:
import os
import random

from tqdm import tqdm
from datasets import Dataset, load_dataset, load_from_disk, concatenate_datasets


In [ ]:
mathverse = load_dataset("AI4Math/MathVerse", "testmini")['testmini']

# Only keep the required columns and rename them using map
def process_mathverse(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': example['answer'],
        'images': [example['image']],
        'source': 'mathverse'
    }

mathverse = mathverse.map(process_mathverse, remove_columns=mathverse.column_names, num_proc=os.cpu_count())

In [ ]:
logicvista = load_dataset("lscpku/LogicVista")['test']

# Only keep the required columns and rename them using map
def process_logicvista(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': example['answer'],
        'images': [example['image']],
        'source': 'logicvista'
    }

logicvista = logicvista.map(process_logicvista, remove_columns=logicvista.column_names, num_proc=os.cpu_count())

In [ ]:
geo3k = load_dataset("hiyouga/geometry3k")['test']

def process_geo3k(example):
    return {
        'problem': example['problem'],
        'answer': example['answer'],
        'images': example['images'],
        'source': 'geo3k'
    }

geo3k = geo3k.map(process_geo3k, remove_columns=geo3k.column_names, num_proc=os.cpu_count())

In [ ]:
wemath = load_dataset("We-Math/We-Math")['testmini']

def process_wemath(example):
    return {
        'problem': f"<image>{example['question']}\n\n{example['option']}",
        'answer': example['answer'],
        'images': [example['image_path']],
        'source': 'wemath'
    }

wemath = wemath.map(process_wemath, remove_columns=wemath.column_names, num_proc=os.cpu_count())

In [ ]:
mathvista = load_dataset("AI4Math/MathVista")['testmini']

def process_mathvista(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': example['answer'],
        'images': [example['decoded_image']],
        'source': 'mathvista'
    }

mathvista = mathvista.map(process_mathvista, remove_columns=mathvista.column_names, num_proc=os.cpu_count())

In [7]:
mmmu_pro = load_dataset("MMMU/MMMU_Pro", 'standard (4 options)')['test']

def filter_single_image(example):
    # Only keep if image_1 is not empty and all other image columns are empty
    return (
        example['image_1'] and
        all(not example[f'image_{i}'] for i in range(2, 8))
    )

def process_mmmu_pro(example):
    options = eval(example['options'])
    option_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']  # extend as needed
    formatted_options = [f"{label}. {opt}" for label, opt in zip(option_labels, options)]
    prompt = example['question'].replace("<image 1>", "<image>") + "\n\n" + "\n".join(formatted_options)
    return {
        'problem': prompt,
        'answer': example['answer'],
        'images': [example['image_1']],
        'source': 'mmmu_pro'
    }

mmmu_pro = mmmu_pro.filter(filter_single_image, num_proc=os.cpu_count())
mmmu_pro = mmmu_pro.map(process_mmmu_pro, remove_columns=mmmu_pro.column_names, num_proc=os.cpu_count())

In [8]:
mathvision = load_dataset("MathLLMs/MathVision")['testmini']

def process_mathvision(example):
    question = example['question'].replace("<image1>", "")
    question = f"<image>{question}"
    return {
        'problem': question,
        'answer': example['answer'],
        'images': [example['decoded_image']],
        'source': 'mathvision'
    }

mathvision = mathvision.map(process_mathvision, remove_columns=mathvision.column_names, num_proc=os.cpu_count())


In [9]:
mmstar = load_dataset("Lin-Chen/MMStar")['val']

def process_mmstar(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': example['answer'],
        'images': [example['image']],
        'source': 'mmstar'
    }

mmstar = mmstar.map(process_mmstar, remove_columns=mmstar.column_names, num_proc=os.cpu_count())


In [10]:
hallbench = load_dataset("lmms-lab/HallusionBench")['image']

def process_hallbench(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': 'yes' if example['gt_answer'] == 1 else 'no',
        'images': [example['image']],
        'source': 'hallbench'
    }

hallbench = hallbench.map(process_hallbench, remove_columns=hallbench.column_names, num_proc=os.cpu_count())


In [11]:
mmvet = load_dataset("lmms-lab/MMVet")['test']

def process_mmvet(example):
    return {
        'problem': f"<image>{example['question']}",
        'answer': example['answer'],
        'images': [example['image']],
        'source': 'mmvet'
    }

mmvet = mmvet.map(process_mmvet, remove_columns=mmvet.column_names, num_proc=os.cpu_count())


num_proc must be <= 218. Reducing num_proc to 218 for dataset of size 218.


In [12]:
# List of all datasets
datasets = [mathverse, logicvista, geo3k, wemath, mathvista, mmmu_pro, mathvision, mmstar, hallbench, mmvet]

# To ensure near-equal number of samples from each, 
# we take the minimum length across datasets as the base.
# Optionally, use a fixed number (e.g., 500) if they're large.

desired_n = min(len(d) for d in datasets if len(d) > 0)
# Optionally, cap 'desired_n' if very large:
desired_n = min(500, desired_n)  # adjust 500 as needed

selected_examples = []
for dataset in datasets:
    n_avail = len(dataset)
    n_samples = min(desired_n, n_avail)
    if n_samples == 0:
        continue
    indices = random.sample(range(n_avail), n_samples)
    selected_examples.append(dataset.select(indices))

all_val_datasets = concatenate_datasets(selected_examples)
all_val_datasets = all_val_datasets.shuffle(seed=0)


In [13]:
# Remove examples where "<image>" appears more than once in the 'problem' field
def has_single_image_token(example):
    return example['problem'].count("<image>") == 1

all_val_datasets = all_val_datasets.filter(has_single_image_token, num_proc=os.cpu_count())

Filter (num_proc=240):   0%|          | 0/2180 [00:00<?, ? examples/s]

In [14]:
# Select 1000 examples from the dataset (or as many as available if <1000)
num_val = min(1000, len(all_val_datasets))
val1k = all_val_datasets.select(random.sample(range(len(all_val_datasets)), num_val))
val1k.push_to_hub("xytian1008/MUPO-Thinker-val1k", split="test")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/xytian1008/MUPO-Thinker-val1k/commit/fe13b6be97aafe8b280e6b6211879fb13d150d7a', commit_message='Upload dataset', commit_description='', oid='fe13b6be97aafe8b280e6b6211879fb13d150d7a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/xytian1008/MUPO-Thinker-val1k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='xytian1008/MUPO-Thinker-val1k'), pr_revision=None, pr_num=None)